In [ ]:
!pip install grad-cam

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
from PIL import Image
from collections import Counter
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
import numpy as np
import torch
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

torch.backends.cudnn.benchmark = True


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
MAG = "400X"
DATA_ROOT = "/kaggle/input/breakhis/BreaKHis_v1//BreaKHis_v1/histology_slides/breast"

BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 1e-4
PATIENCE = 25

CLASS_NAMES = [
    "adenosis",
    "fibroadenoma",
    "phyllodes_tumor",
    "tubular_adenoma",
    "ductal_carcinoma",
    "lobular_carcinoma",
    "mucinous_carcinoma",
    "papillary_carcinoma"
]

class_to_idx = {cls: i for i, cls in enumerate(CLASS_NAMES)}

In [ ]:
def load_breakhis(mag="400X"):
    samples = []

    for root, _, files in os.walk(DATA_ROOT):
        if not root.endswith(mag):
            continue

        parts = root.split(os.sep)

        cls = None
        for c in CLASS_NAMES:
            if c in parts:
                cls = c
                break

        if cls is None:
            continue

        label = class_to_idx[cls]

        for f in files:
            if f.lower().endswith(".png"):
                samples.append((os.path.join(root, f), label))

    return samples

In [ ]:
class BreakHisDataset(Dataset):

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        try:
            img = Image.open(img_path).convert("RGB")

            if self.transform:
                img = self.transform(img)

            return img, label

        except:
            return torch.zeros((3, 224, 224)), label

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((230, 230)),
    transforms.RandomRotation(1),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [ ]:
full_samples = load_breakhis(MAG)

if not full_samples:
    raise ValueError("Không tìm thấy ảnh!")

print("Total samples:", len(full_samples))

In [ ]:
from sklearn.model_selection import train_test_split

train_samples, temp_samples = train_test_split(
    full_samples,
    test_size=0.2,
    stratify=[s[1] for s in full_samples],
    random_state=42
)

val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=0.5,
    stratify=[s[1] for s in temp_samples],
    random_state=42
)

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_samples))

In [ ]:

train_targets = [s[1] for s in train_samples]
class_sample_count = Counter(train_targets)

weights = 1. / torch.tensor(
    [class_sample_count[i] for i in range(len(CLASS_NAMES))],
    dtype=torch.float
)

samples_weights = torch.tensor([weights[t] for t in train_targets])
sampler = WeightedRandomSampler(samples_weights, len(samples_weights))


In [ ]:
train_ds = BreakHisDataset(train_samples, train_tf)
val_ds   = BreakHisDataset(val_samples, val_tf)
test_ds  = BreakHisDataset(test_samples, val_tf)  

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,   
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(   
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
def plot_distribution(counts, title):

    labels = CLASS_NAMES
    values = [counts[i] for i in range(len(labels))]

    plt.figure(figsize=(10,4))
    plt.bar(labels, values)
    plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.show()


In [ ]:
orig_counter = Counter(train_targets)
plot_distribution(orig_counter, "Train set distribution (before sampler)")

In [ ]:
sampled_indices = list(iter(sampler))
sampled_labels = [train_targets[i] for i in sampled_indices]
sampled_counter = Counter(sampled_labels)

plot_distribution(sampled_counter, "Train set distribution (after WeightedRandomSampler)")


In [ ]:
train_ds = BreakHisDataset(train_samples, train_tf)
val_ds   = BreakHisDataset(val_samples, val_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.mlp(self.avg_pool(x))
        max_ = self.mlp(self.max_pool(x))
        return self.sigmoid(avg + max_)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg, max_], dim=1)
        x = self.conv(x)
        return self.sigmoid(x)


class CBAM(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.ca = ChannelAttention(in_channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

In [ ]:
class ConvNeXt_CBAM(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.backbone = convnext_tiny(
            weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        )

        last_channels = self.backbone.classifier[2].in_features

        self.cbam = CBAM(last_channels)

        self.backbone.classifier = nn.Identity()

        self.classifier = nn.Sequential(
            nn.LayerNorm(last_channels),
            nn.Linear(last_channels, num_classes)
        )

    def forward(self, x):
        x = self.backbone.features(x)   

        x = self.cbam(x)              

        x = x.mean(dim=[2,3])           
        x = self.classifier(x)
        return x

In [ ]:
model = ConvNeXt_CBAM(num_classes=8).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.15)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-2
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.3
)

In [ ]:
def calculate_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    p, r, f, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    return acc, p, r, f

In [ ]:
train_losses = []
val_losses   = []
train_accs   = []
val_accs     = []

best_f1 = 0.0
best_epoch = 0
best_state = None
best_metrics = None
best_pred = None  
patience_counter = 0

import copy  

print(f"{'Epoch':^6} | {'Train Loss':^10} | {'Val Loss':^9} | {'Val Acc':^9} | {'Val Prec':^9} | {'Val Rec':^9} | {'Val F1':^9} | Status")
print("-" * 110)

for epoch in range(1, NUM_EPOCHS + 1):

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        scheduler.step()

        train_loss += loss.item()

        preds = outputs.argmax(1)

        train_correct += (preds == labels).sum().item()

        train_total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)

    train_acc = train_correct / train_total

    model.eval()

    y_true, y_pred = [], []

    val_loss = 0.0

    with torch.no_grad():

        for imgs, labels in val_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    val_loss /= len(val_loader)

    val_acc, val_prec, val_rec, val_f1 = calculate_metrics(y_true, y_pred)

    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    status = ""

    if val_f1 > best_f1:

        best_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        best_metrics = (val_acc, val_prec, val_rec, val_f1)
        best_pred = (y_true.copy(), y_pred.copy())

        patience_counter = 0
        status = "BEST"

    else:

        patience_counter += 1

    print(f"{epoch:^6} | {avg_train_loss:^10.4f} | {val_loss:^9.4f} | "
          f"{val_acc*100:^8.2f}% | {val_prec*100:^8.2f}% | "
          f"{val_rec*100:^8.2f}% | {val_f1*100:^8.2f}% | {status}")

    if patience_counter >= PATIENCE:

        print(f"\nEarly stopping at epoch {epoch}")
        break

In [ ]:
model.load_state_dict(best_state)
model.eval()

y_true_test, y_pred_test = [], []

with torch.no_grad():
    for imgs, labels in test_loader:

        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        y_true_test.extend(labels.cpu().numpy())
        y_pred_test.extend(preds.cpu().numpy())

In [ ]:
test_acc, test_prec, test_rec, test_f1 = calculate_metrics(
    y_true_test, y_pred_test
)

In [ ]:
print("\nFINAL TEST RESULT")
print(f" Accuracy  : {test_acc*100:.2f}%")
print(f" Precision : {test_prec*100:.2f}%")
print(f" Recall    : {test_rec*100:.2f}%")
print(f" F1-score  : {test_f1*100:.2f}%")

In [ ]:
y_true, y_pred = best_pred
print("\nClassification Report (TEST):")
print(classification_report(y_true_test, y_pred_test, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true_test, y_pred_test)

disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(xticks_rotation=45)

plt.title("Confusion Matrix (Test Set)")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Train vs Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Val Accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# GRAD-CAM SETUP
# =========================

features = []
gradients = []

def forward_hook(module, input, output):
    features.clear()
    features.append(output)

def backward_hook(module, grad_input, grad_output):
    gradients.clear()
    gradients.append(grad_output[0])

# 🔥 Layer đúng cho ConvNeXt
target_layer = model.backbone.features[-1]

target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)


# =========================
# CORE GRAD-CAM
# =========================

def grad_cam_single(img, class_idx):
    model.eval()

    img = img.unsqueeze(0).to(device)

    features.clear()
    gradients.clear()

    output = model(img)

    model.zero_grad()
    output[0, class_idx].backward(retain_graph=True)

    fmap = features[0]   # [1, C, H, W]
    grad = gradients[0]  # [1, C, H, W]

    weights = grad.mean(dim=(2, 3), keepdim=True)

    cam = (weights * fmap).sum(dim=1).squeeze()

    cam = cam.detach().cpu().numpy()
    cam = np.maximum(cam, 0)

    if cam.max() != 0:
        cam = cam / cam.max()

    cam = cv2.resize(cam, (224, 224))

    return cam


# =========================
# UTILS
# =========================

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    img = img.permute(1,2,0).cpu().numpy()
    img = std * img + mean
    img = np.clip(img, 0, 1)

    return img


# =========================
# VISUALIZE 8 CLASSES
# =========================

def visualize_8cams(img, label):
    cams = []

    for i in range(len(CLASS_NAMES)):
        cams.append(grad_cam_single(img, i))

    img_np = denormalize(img)

    plt.figure(figsize=(16,10))

    for i, cam in enumerate(cams):
        plt.subplot(2,4,i+1)

        plt.imshow(img_np)
        plt.imshow(cam, cmap='jet', alpha=0.5)

        if i == label:
            plt.title(f"{CLASS_NAMES[i]} ⭐", color='red')
        else:
            plt.title(CLASS_NAMES[i])

        plt.axis('off')

    plt.suptitle(f"Ground Truth: {CLASS_NAMES[label]}")
    plt.tight_layout()
    plt.show()


# =========================
# VISUALIZE PREDICTED CLASS
# =========================

def visualize_predicted_cam(img):
    model.eval()

    img_input = img.unsqueeze(0).to(device)

    output = model(img_input)
    pred_class = torch.argmax(output, dim=1).item()

    cam = grad_cam_single(img, pred_class)

    img_np = denormalize(img)

    plt.figure(figsize=(5,5))

    plt.imshow(img_np)
    plt.imshow(cam, cmap='jet', alpha=0.5)

    plt.title(f"Predicted: {CLASS_NAMES[pred_class]}")
    plt.axis('off')
    plt.show()


# =========================
# TEST SAMPLE (TEST SET)
# =========================

imgs, labels = next(iter(test_loader))

img = imgs[0].cpu()
label = labels[0].item()

visualize_8cams(img, label)
visualize_predicted_cam(img)